In [4]:
# teste

import sys, jsonschema
print(sys.executable)
print(sys.version)
print("jsonschema:", jsonschema.__version__)

c:\Users\bruna.sousa\Documents\ELO\git\elocartoes-compraonline\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
jsonschema: 4.25.1


C:\Users\bruna.sousa\AppData\Local\Temp\ipykernel_3344\470255447.py:6: DeprecationWarning: Accessing jsonschema.__version__ is deprecated and will be removed in a future release. Use importlib.metadata directly to query for jsonschema's version.
  print("jsonschema:", jsonschema.__version__)


In [5]:
# --- Configuração dos caminhos dos arquivos e opção para salvar exemplos ---

import json, uuid, random, string
from pathlib import Path
from datetime import datetime, timezone
from jsonschema import Draft202012Validator

# mude o caminho abaixo para o diretório onde estão os schemas e onde quer salvar os exemplos
BASE = Path(r"C:\Users\bruna.sousa\Documents\ELO\git\elocartoes-compraonline\schemas")
REQ_SCHEMA_PATH  = BASE / "fraud_scoring_request.schema.json"
RESP_SCHEMA_PATH = BASE / "fraud_scoring_response.schema.json"
EXAMPLES_DIR     = BASE / "examples"

SAVE_FILES = True  # mude para False se não quiser gravar os JSONs 

EXAMPLES_DIR.mkdir(exist_ok=True)
print("Schemas em:", BASE)
print("Exemplos em:", EXAMPLES_DIR)


Schemas em: C:\Users\bruna.sousa\Documents\ELO\git\elocartoes-compraonline\schemas
Exemplos em: C:\Users\bruna.sousa\Documents\ELO\git\elocartoes-compraonline\schemas\examples


In [6]:
# ---  Carrega schemas reais da pasta ---
def load_json(p: Path):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

req_schema  = load_json(REQ_SCHEMA_PATH)
resp_schema = load_json(RESP_SCHEMA_PATH)

# mostra as chaves principais pra conferência rápida
req_required = req_schema.get("required", [])
resp_required = resp_schema.get("required", [])
print("Request required:", req_required)
print("Response required:", resp_required)


Request required: ['transaction_id', 'card_number', 'purchase_amount', 'purchase_currency', 'purchase_date', 'device_channel']
Response required: ['transaction_id', 'score', 'risk_band', 'recommendation', 'model_version', 'timestamp', 'latency_ms', 'trace_id']


In [7]:
# --- Funções para gerar valores de exemplo para cada campo, usando heurísticas simples ---

def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def yyyymmddhhmmss():
    return datetime.now().strftime("%Y%m%d%H%M%S")

def rand_digits(n):
    return "".join(random.choice(string.digits) for _ in range(n))

# valores especiais para alguns campos comuns
SPECIAL_VALUES = {
    "addr_match": "Y",                     # regex [YN]
    "browser_color_depth": "24",           # curto (ex.: 24 bits)
    "purchase_exponent": "2",              # \d{1}
    "purchase_instal_data": "0",           # curto (use "01" se seu maxLength==2)
    "three_ds_trans_status_reason": "01",  # \d{2}
}

# tenta adivinhar um valor razoável para o campo, baseado no nome e tipo
def guess_value_for_field(name: str, prop: dict):
    name_l = name.lower()
    typ = prop.get("type")

    # 1) se houver valor especial definido para o campo, usar
    if name in SPECIAL_VALUES:
        return SPECIAL_VALUES[name]

    # 2) se o campo for um enum (lista de valores permitidos), usa o primeiro valor da lista
    if "enum" in prop and isinstance(prop["enum"], list) and prop["enum"]:
        return prop["enum"][0]

    # 3) heurísticas por nome 
    if "transaction_id" in name_l:
        import uuid; return str(uuid.uuid4())
    if name_l in ("card_number","pan","acctnumber"):
        return "4111111111111111"
    if "amount" in name_l:
        return "123.45" if typ == "string" else 123.45
    if "currency" in name_l:
        return "986" if typ == "string" else 986
    if name_l in ("purchase_date","purchasedate"):
        from datetime import datetime; return datetime.now().strftime("%Y%m%d%H%M%S")
    if "event_ts" in name_l or ("timestamp" in name_l and typ=="string"):
        from datetime import datetime, timezone; return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    if "device_channel" in name_l:
        return "BROWSER"
    if name_l == "mcc":
        return "5411"
    if "country" in name_l and "code" in name_l:
        return "076"
    if "browser_ip" in name_l or name_l.endswith("ip"):
        return "203.0.113.10"
    if "three_ds_trans_status" in name_l or "transstatus" in name_l:
        return "Y"
    if "three_ds_eci" in name_l or name_l == "eci":
        return "05"
    if "email" in name_l:
        return "cliente@example.com"
    if name_l.endswith("bin"):
        return "520123"
    if name_l.endswith("last4"):
        return "7741"
    if name_l == "trace_id":
        import uuid; return str(uuid.uuid4())
    if "latency" in name_l and typ == "integer":
        return 180
    if name_l == "environment":
        return "dev"

    # fallback por tipo
    if typ == "string":  return "value"
    if typ == "number":  return 1.0
    if typ == "integer": return 1
    if typ == "boolean": return True
    if typ == "array":   return []
    if typ == "object":  return {}
    return "value"


In [8]:
# --- Gera exemplos REQUEST com base no schema real ---

def make_request_example(schema: dict, only_required=False):
    props = schema.get("properties", {})
    required = schema.get("required", [])
    data = {}

    fields = required if only_required else list(props.keys() or [])
    for name in fields:
        prop = props.get(name, {})
        data[name] = guess_value_for_field(name, prop)

    return data

request_valid = make_request_example(req_schema, only_required=False)
request_missing_optional = make_request_example(req_schema, only_required=True)

print("request_valid keys:", list(request_valid.keys())[:10], "...")
print("request_missing_optional keys:", list(request_missing_optional.keys()))


request_valid keys: ['transaction_id', 'card_number', 'purchase_amount', 'purchase_currency', 'purchase_exponent', 'purchase_date', 'device_channel', 'mcc', 'merchant_country_code', 'acquirer_merchant_id'] ...
request_missing_optional keys: ['transaction_id', 'card_number', 'purchase_amount', 'purchase_currency', 'purchase_date', 'device_channel']


In [9]:
# --- Gera exemplos RESPONSE com base no schema real ---

def make_response_success(schema: dict):
    props = schema.get("properties", {})
    required = schema.get("required", [])
    data = {}

 
    # preencher required primeiro
    for name in required:
        prop = props.get(name, {})
        data[name] = guess_value_for_field(name, prop)

    # alguns campos típicos (se existirem)
    if "reason_codes" in props:
        data["reason_codes"] = [
            {"code":"ip_risk","description":"IP com histórico recente","impact":0.21},
            {"code":"mcc","description":"MCC de risco","impact":0.12},
        ]
    if "timestamp" in props:
        data["timestamp"] = now_iso()
    if "score" in props:
        data["score"] = 0.73
    if "risk_band" in props:
        data["risk_band"] = "HIGH"
    if "recommendation" in props:
        data["recommendation"] = "REVIEW"
    if "model_version" in props:
        data["model_version"] = "v1.0.0"
    if "latency_ms" in props:
        data["latency_ms"] = 180
    if "environment" in props:
        data["environment"] = "dev"

    return data

def make_response_fallback(schema: dict):
    data = make_response_success(schema)
    # ajustar para cenário de fallback/erro
    data["score"] = 0.50
    if "risk_band" in data:
        data["risk_band"] = "MEDIUM"
    if "recommendation" in data:
        data["recommendation"] = "REVIEW"
    # warnings/erro, se previstos no schema
    if "warnings" in schema.get("properties", {}):
        data["warnings"] = [{"code":"MISSING_OPTIONAL","message":"browser_ip ausente; usando default"}]
    if "error" in schema.get("properties", {}):
        data["error"] = {
            "code":"MODEL_TIMEOUT",
            "message":"Timeout ao chamar endpoint",
            "detail":"SageMaker p95 acima do limite",
            "fallback_applied": True
        }
    if "model_version" in data:
        data["model_version"] = "v0.0.1"   

    if "latency_ms" in data:
        data["latency_ms"] = 480
    return data

response_success  = make_response_success(resp_schema)
response_fallback = make_response_fallback(resp_schema)

list(response_success.keys())[:10], list(response_fallback.keys())[:10]


(['transaction_id',
  'score',
  'risk_band',
  'recommendation',
  'model_version',
  'timestamp',
  'latency_ms',
  'trace_id',
  'reason_codes',
  'environment'],
 ['transaction_id',
  'score',
  'risk_band',
  'recommendation',
  'model_version',
  'timestamp',
  'latency_ms',
  'trace_id',
  'reason_codes',
  'environment'])

In [10]:
# --- Validadores e execução dos testes ---

def validate_instance(instance: dict, schema: dict, label: str):
    v = Draft202012Validator(schema)
    errors = sorted(v.iter_errors(instance), key=lambda e: e.path)
    if errors:
        print(f" {label}: inválido")
        for e in errors[:10]:
            loc = ".".join(map(str, e.path)) or "<root>"
            print(f"   - {loc}: {e.message}")
    else:
        print(f" {label}: válido")

print("Request validations:")
validate_instance(request_valid, req_schema, "request_valid")
validate_instance(request_missing_optional, req_schema, "request_missing_optional")

print("\nResponse validations:")
validate_instance(response_success,  resp_schema, "response_success")
validate_instance(response_fallback, resp_schema, "response_fallback")


Request validations:
 request_valid: válido
 request_missing_optional: válido

Response validations:
 response_success: válido
 response_fallback: válido


In [11]:
# --- Salva os exemplos em schemas/examples/ ---

def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print("salvo:", path)

if SAVE_FILES:
    save_json(request_valid,            EXAMPLES_DIR / "request_valid.json")
    save_json(request_missing_optional, EXAMPLES_DIR / "request_missing_optional.json")
    save_json(response_success,         EXAMPLES_DIR / "response_success.json")
    save_json(response_fallback,        EXAMPLES_DIR / "response_fallback.json")


salvo: C:\Users\bruna.sousa\Documents\ELO\git\elocartoes-compraonline\schemas\examples\request_valid.json
salvo: C:\Users\bruna.sousa\Documents\ELO\git\elocartoes-compraonline\schemas\examples\request_missing_optional.json
salvo: C:\Users\bruna.sousa\Documents\ELO\git\elocartoes-compraonline\schemas\examples\response_success.json
salvo: C:\Users\bruna.sousa\Documents\ELO\git\elocartoes-compraonline\schemas\examples\response_fallback.json
